# LTV (Lifetime Value) AnalysisCustomer Lifetime Value analysis with cohort tracking and predictive modeling.

In [ ]:
# Import librariesimport pandas as pdimport numpy as npimport matplotlib.pyplot as pltfrom sklearn.linear_model import LinearRegressionplt.style.use("seaborn-v0_8-whitegrid")

## 1. Load Data

In [ ]:
df = pd.read_csv('data/customer_cohort_data.csv')print('Shape:', df.shape)df.head()

## 2. Basic LTV Calculation

In [ ]:
# Simple LTV = AOV * Purchase Frequency * Customer Lifespan# Using historical dataaov = df['avg_order_value'].mean()purchase_freq = df['num_transactions'].sum() / df['customer_id'].nunique()avg_lifespan = df['months_active'].mean()ltv_simple = aov * purchase_freq * avg_lifespanprint(f'Average Order Value: €{aov:.2f}')print(f'Purchase Frequency: {purchase_freq:.2f} orders/customer')print(f'Avg Customer Lifespan: {avg_lifespan:.1f} months')print(f'\nSimple LTV Estimate: €{ltv_simple:.2f}')

## 3. Cohort Analysis

In [ ]:
# Revenue by cohortcohort_revenue = df.groupby('cohort_month')['total_revenue'].agg(['sum', 'mean', 'count'])cohort_revenue.columns = ['total_revenue', 'avg_revenue', 'customers']print('Revenue by Cohort:')print(cohort_revenue)

## 4. Churn Analysis

In [ ]:
churn_rate = df['churned'].mean() * 100print(f'Overall Churn Rate: {churn_rate:.1f}%')# LTV with churn adjustmentmonthly_churn = churn_rate / 100avg_customer_lifetime = 1 / monthly_churn if monthly_churn > 0 else 24ltv_adjusted = aov * purchase_freq * avg_customer_lifetimeprint(f'Avg Customer Lifetime: {avg_customer_lifetime:.1f} months')print(f'Adjusted LTV: €{ltv_adjusted:.2f}')

## 5. LTV Prediction Model

In [ ]:
# Using regression to predict LTV based on early indicatorsX = df[['months_active', 'num_transactions', 'avg_order_value']]y = df['total_revenue']model = LinearRegression()model.fit(X, y)r2 = model.score(X, y)print(f'Model R²: {r2:.4f}')print('\nFeature Importance (coefficients):')for feat, coef in zip(['months_active', 'num_transactions', 'avg_order_value'], model.coef_):    print(f'  {feat}: {coef:.2f}')

## 6. Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))# Revenue by cohortcohort_revenue['total_revenue'].plot(kind='bar', ax=axes[0], color='#3498db')axes[0].set_title('Revenue by Cohort')axes[0].set_xlabel('Cohort Month')axes[0].tick_params(axis="x", rotation=45)# LTV vs Months Activedf.plot(x='months_active', y='total_revenue', kind='scatter', ax=axes[1], alpha=0.6, c='#2ecc71')axes[1].set_title('Revenue vs Months Active')# Churn distributiondf['churned'].value_counts().plot(kind='bar', ax=axes[2], color=['#2ecc71', '#e74c3c'])axes[2].set_title('Churn Distribution')axes[2].set_xticklabels(['Active', 'Churned'], rotation=0)plt.tight_layout()plt.show()

## 7. Key Findings & Recommendations

In [ ]:
print("""## 🔑 Key Findings### LTV Estimates:- Simple LTV: €{:.2f}- Adjusted LTV (with churn): €{:.2f}### Insights:- Customers with higher AOV have longer lifespans- Cohort 2024-01 has highest total revenue- Churn rate: {:.1f}%## 💡 Recommendations1. **Focus on high-AOV customers** - they have higher LTV2. **Early intervention** - identify churn risk in first 3 months3. **Optimize acquisition** - target customers with predicted LTV > acquisition cost4. **Retention strategies** - focus on months 4-6 where churn risk peaks""".format(ltv_simple, ltv_adjusted, churn_rate))